In [ ]:
import scanpy as sc
import plotnine as gg
import matplotlib.pyplot as plt

plt.rcParams["svg.fonttype"] = "none"

In [ ]:
adata = sc.read_h5ad("/workspace/experiments/04302026_analysis_data_v2/adata_de122_lce75_merged.h5ad")

In [ ]:
sc.pl.umap(adata, color=["leiden_0.5"])

In [ ]:
adata.obs["is_control"] = adata.obs["leiden_0.5"].isin(["0", "1", "2", "3", "4", "5", "6"])

In [ ]:
sc.pl.umap(adata, color=["is_control"])

In [ ]:
adata_case = adata[~adata.obs["is_control"]].copy()
sc.pp.highly_variable_genes(adata_case, n_top_genes=1000, flavor="seurat_v3")
adata_case = adata_case[:, adata_case.var["highly_variable"]].copy()

In [ ]:
import scvi

In [ ]:
scvi.model.SCVI.setup_anndata(adata_case, categorical_covariate_keys=["rt_bc", "experiment"], layer="reads")
model = scvi.model.SCVI(adata_case)
model.train()

In [ ]:
latent = model.get_latent_representation()
adata_case.obsm["latent"] = latent

In [ ]:
# sc.tl.pca(adata_case, layer="lognormalized", n_comps=50)
# sc.pp.neighbors(adata_case, n_neighbors=15)
sc.pp.neighbors(adata_case, use_rep="latent", n_neighbors=15)
sc.tl.umap(adata_case, min_dist=0.5)
sc.tl.leiden(adata_case)

adata_case.obs["WITH_PHENOTYPE_UMAP1"] = adata_case.obsm["X_umap"][:, 0]
adata_case.obs["WITH_PHENOTYPE_UMAP2"] = adata_case.obsm["X_umap"][:, 1]

(
    gg.ggplot(adata_case.obs, gg.aes(x="WITH_PHENOTYPE_UMAP1", y="WITH_PHENOTYPE_UMAP2", color="leiden"))
    + gg.geom_point(size=1.0, stroke=0.0)
    # + gg.theme_classic()
    # + PLOTNINE_DEFAULT_THEME_2
    + gg.labs(
        title="scVI latent space (batch correction)"
    )
    + gg.guides(color=gg.guide_legend(override_aes={"size": 5}))
)


In [ ]:
adata_case[adata_case.obs["leiden"] == leiden_cluster].obs

In [ ]:
for leiden_cluster in range(30):
    print("Cluster: ", leiden_cluster)
    obs_subset = adata_case[adata_case.obs["leiden"] == str(leiden_cluster)].obs
    vals = obs_subset["target"].value_counts().head(50)
    print(vals)

In [ ]:
renamer = {
  "0": "Uncharacterized / no-phenotype catch-all (weak)",
  "1": "Mixed metabolic / transport functions (weak)",
  "2": "Stringent response / tRNA metabolism & aminoacyl-tRNA synthetases",
  "3": "Stress response / iron-sulfur cluster regulation & stringent starvation (weak)",
  "4": "Tol-Pal outer membrane integrity / peptidoglycan (UDP-GlcNAc) biosynthesis",
  "5": "Ribosome biogenesis / rRNA processing & modification",
  "6": "Membrane biogenesis: fatty acid & phospholipid synthesis, MreB cell shape",
  "7": "Central metabolism: CoA biosynthesis, fatty acid synthesis, NAD biosynthesis, glycolysis",
  "8": "Translation: ribosomal proteins & translation initiation/elongation/termination factors",
  "9": "Isoprenoid / cofactor biosynthesis: heme, ubiquinone (CoQ), riboflavin, MEP pathway",
  "10": "Aminoacyl-tRNA synthetases / tRNA charging",
  "11": "Central carbon metabolism: TCA cycle, glycolysis, PTS system",
  "12": "Zinc homeostasis / metal uptake (zur/znu) (weak)",
  "13": "DNA replication & nucleoid organization (DnaE, SeqA, Fis, HupA)",
  "14": "Outer membrane biogenesis: LPS/lipid A synthesis, BAM complex, Lol lipoprotein transport, Lpt LPS transport",
  "15": "Cell division (FtsZ ring) / SRP co-translational targeting",
  "16": "ATP synthase (F1Fo complex)",
  "17": "LPS core oligosaccharide biosynthesis (Waa/Rfa pathway)",
  "18": "Enterobactin-mediated iron uptake / Fur regulation (weak – DAP biosynthesis genes co-cluster)",
  "19": "Sec translocon / transcription elongation (SecE–NusG operon)",
  "20": "Phosphate-specific transport & signaling (Pst/PhoU) (weak – GroEL/ES co-cluster)",
  "21": "Cytochrome bo terminal oxidase / aerobic respiration",
  "22": "DNA gyrase / DNA topology (weak – racR prophage gene dominates)",
  "23": "RNase P / tRNA processing (RnpA/RnpB)",
  "24": "Rho-dependent transcription termination",
  "25": "NusA transcription elongation / ribosome maturation (RimP)",
  "26": "Methionine / S-adenosylmethionine (SAM) metabolism (MetJ/MetK)",
  "27": "RNA polymerase omega subunit (RpoZ)",
  "28": "Glucose PTS transport / upper glycolysis (PtsG, Pgi)",
  "29": "CsrA carbon storage / post-transcriptional regulation"
}

In [ ]:

(
    gg.ggplot(adata_case.obs.loc[lambda x: ~x["leiden"].isin(["0", "1",])], gg.aes(x="WITH_PHENOTYPE_UMAP1", y="WITH_PHENOTYPE_UMAP2", color="leiden"))
    + gg.geom_point(size=1.0, stroke=0.0)
    # + gg.theme_classic()
    # + PLOTNINE_DEFAULT_THEME_2
    + gg.labs(
        title="scVI latent space (batch correction)"
    )
    + gg.guides(color=gg.guide_legend(override_aes={"size": 5}))
)


In [ ]:
barcodes_to_keep = adata_case[~adata_case.obs["leiden"].isin(["0", "1"])].obs.index
barcodes_to_keep

In [ ]:
adata_case2 = adata[barcodes_to_keep].copy()
sc.pp.highly_variable_genes(adata_case2, n_top_genes=1000, flavor="seurat_v3")
adata_case2 = adata_case2[:, adata_case2.var["highly_variable"]].copy()

In [ ]:
scvi.model.SCVI.setup_anndata(adata_case2, categorical_covariate_keys=["rt_bc", "experiment"], layer="reads")
model = scvi.model.SCVI(adata_case2)
model.train()

In [ ]:
latent = model.get_latent_representation()
adata_case2.obsm["latent"] = latent

sc.pp.neighbors(adata_case2, use_rep="latent", n_neighbors=15)
sc.tl.umap(adata_case2, min_dist=0.4)
sc.tl.leiden(adata_case2)

adata_case2.obs["WITH_PHENOTYPE_UMAP1"] = adata_case2.obsm["X_umap"][:, 0]
adata_case2.obs["WITH_PHENOTYPE_UMAP2"] = adata_case2.obsm["X_umap"][:, 1]

(
    gg.ggplot(adata_case2.obs, gg.aes(x="WITH_PHENOTYPE_UMAP1", y="WITH_PHENOTYPE_UMAP2", color="leiden"))
    + gg.geom_point(size=1.0, stroke=0.0)
    # + gg.theme_classic()
    # + PLOTNINE_DEFAULT_THEME_2
    + gg.labs(
        title="scVI latent space (batch correction)"
    )
    + gg.guides(color=gg.guide_legend(override_aes={"size": 5}))
)


In [ ]:
for leiden_cluster in range(37):
    print("Cluster: ", leiden_cluster)
    obs_subset = adata_case2[adata_case2.obs["leiden"] == str(leiden_cluster)].obs
    vals = obs_subset["target"].value_counts().head(25)
    print(vals)

In [ ]:
labels = {
  "0": "translation initiation & stress response",
  "1": "amino acid transport & metabolism",
  "2": "ribosome biogenesis",
  "3": "central metabolism",
  "4": "cell envelope integrity",
  "5": "translation",
  "6": "aerobic respiration",
  "7": "nucleoid-associated & uncharacterized",
  "8": "aminoacyl-tRNA synthetases",
  "9": "DNA replication",
  "10": "membrane biogenesis",
  "11": "metabolic regulation",
  "12": "central carbon metabolism",
  "13": "outer membrane biogenesis",
  "14": "heme & ubiquinone biosynthesis",
  "15": "tRNA charging & modification",
  "16": "LPS core biosynthesis",
  "17": "ribosome assembly",
  "18": "ATP synthase",
  "19": "enterobactin iron uptake",
  "20": "transcription termination",
  "21": "cell division",
  "22": "phosphate transport",
  "23": "tRNA processing",
  "24": "transcription elongation",
  "25": "DNA topology",
  "26": "SRP targeting",
  "27": "SAM metabolism",
  "28": "yrfF",
  "29": "rpoZ",
  "30": "tRNA processing",
  "31": "glucose PTS",
  "32": "zinc homeostasis",
  "33": "iscR",
  "34": "csrA",
  "35": "racR",
  "36": "uncharacterized"
}

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
adata_case2.obs["label_auto"] = adata_case2.obs["leiden"].map(labels)

In [ ]:
adata_case2.obs["label_auto"].value_counts()

In [ ]:
import seaborn as sns
import matplotlib.patheffects as pe


# 'husl' or 'tab20' + 'tab20b' + 'tab20c' combined for 37 colors
palette = sns.color_palette("Spectral", 37).as_hex()
adata_case2.uns["label_auto_colors"] = palette  # must match category order

fig, ax = plt.subplots(figsize=(20, 15))
sc.pl.umap(adata_case2, color="label_auto", legend_loc="on data",
           legend_fontsize="small", ax=ax, show=False, s=50)
for text in ax.texts:
    text.set_path_effects([
        pe.withStroke(linewidth=3, foreground="white")
    ])
plt.savefig("umap_case.svg")

In [ ]:
adata_case2.uns["label_auto_colors"] = None

In [ ]:
adata_case2.write_h5ad("adata_de122_lce75_merged.case.h5ad")